In [ ]:
from datetime import datetime as dt

track_data = {
    "datetime": [
        "2014-09-10 18:00", "2014-09-11 00:00", "2014-09-11 06:00", "2014-09-11 12:00",
        "2014-09-11 18:00", "2014-09-12 00:00", "2014-09-12 06:00", "2014-09-12 12:00",
        "2014-09-12 18:00", "2014-09-13 00:00", "2014-09-13 06:00", "2014-09-13 12:00",
        "2014-09-13 18:00", "2014-09-14 00:00", "2014-09-14 06:00", "2014-09-14 12:00",
        "2014-09-14 18:00", "2014-09-15 00:00", "2014-09-15 06:00", "2014-09-15 12:00",
        "2014-09-15 18:00", "2014-09-16 00:00", "2014-09-16 06:00", "2014-09-16 12:00",
        "2014-09-16 18:00", "2014-09-17 00:00", "2014-09-17 06:00", "2014-09-17 12:00",
        "2014-09-17 18:00", "2014-09-18 00:00", "2014-09-18 06:00", "2014-09-18 12:00",
        "2014-09-18 18:00", "2014-09-19 00:00", "2014-09-19 06:00", "2014-09-19 12:00",
        "2014-09-19 18:00", "2014-09-20 00:00", "2014-09-20 06:00", "2014-09-20 12:00",
        "2014-09-20 18:00"
    ],
    "lat": [
        13.7, 14.2, 14.9, 15.8, 16.5, 17.2, 17.7, 18.3, 18.9, 19.6, 20.4, 21.4, 22.3,
        23.0, 23.7, 24.4, 25.1, 25.8, 26.5, 27.0, 27.7, 28.5, 29.5, 30.6, 31.7,
        32.9, 34.3, 35.7, 37.2, 38.3, 39.3, 39.9, 39.9, 39.9, 39.9, 39.5, 39.5,
        39.4, 39.3, 39.1, 38.8
    ],
    "lon": [
        -33.2, -34.4, -35.5, -36.5, -37.7, -39.2, -40.8, -42.3, -43.6, -44.8,
        -45.7, -46.4, -47.0, -47.9, -49.0, -50.3, -51.5, -52.7, -54.0, -55.1,
        -56.1, -56.7, -57.3, -57.8, -57.7, -57.0, -56.0, -54.5, -52.4, -49.5,
        -46.7, -44.1, -42.1, -40.1, -38.9, -38.1, -37.4, -36.6, -35.7, -34.4, -33.0
    ],
    "pressure": [
        1009, 1009, 1009, 1008, 1007, 1006, 1005, 1000, 999, 998, 997, 996, 995,
        994, 991, 985, 980, 975, 970, 966, 962, 960, 956, 955, 958, 959, 958,
        957, 955, 958, 964, 969, 973, 980, 988, 994, 1001, 1004, 1006, 1008, 1010
    ],
    "wind": [
        25, 25, 25, 30, 30, 35, 35, 40, 40, 40, 45, 45, 50, 55, 60, 70, 75, 80,
        85, 90, 95, 95, 100, 105, 95, 85, 85, 80, 80, 75, 70, 70, 65, 60, 55,
        45, 40, 35, 30, 30, 25
    ],
    "stage": [
        "low", "low", "low", "tropical depression", "tropical depression",
        "tropical storm", "tropical storm", "tropical storm", "tropical storm",
        "tropical storm", "tropical storm", "tropical storm", "tropical storm",
        "tropical storm", "tropical storm", "hurricane", "hurricane", "hurricane",
        "hurricane", "hurricane", "hurricane", "hurricane", "hurricane",
        "hurricane", "hurricane", "hurricane", "hurricane", "hurricane",
        "hurricane", "hurricane", "hurricane", "hurricane", "hurricane",
        "tropical storm", "tropical storm", "tropical storm", "low", "low",
        "low", "low", "low"
    ]
}


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from geopy.distance import great_circle
import re

# --- File path ---
cat_file = "DATA/20140915H1.cat"

# --- Read .cat file into DataFrame ---
rows = []
with open(cat_file, "r") as f:
    for line in f:
        if re.match(r"\s*\d+\s+\d+", line):  # line starts with sequence number and serial
            parts = line.split()
            seq = int(parts[0])
            serial = parts[1]
            date_time = parts[2] + " " + parts[3]
            lat = float(parts[4])
            lon = float(parts[5])
            pressure = float(parts[6])
            comment = " ".join(parts[7:])
            rows.append([seq, serial, date_time, lat, lon, pressure, comment])

df = pd.DataFrame(rows, columns=["Seq", "Serial", "Date/Time", "Lat", "Lon", "Pressure", "Comment"])

# --- Identify center drop (based on comment keyword) ---
center_row = df[df["Comment"].str.contains("center", case=False, na=False)].iloc[0]
center_lat = center_row["Lat"]
center_lon = center_row["Lon"]

# --- Compute distances and bearings relative to the center ---
def calc_offset(lat, lon, center_lat, center_lon):
    # distance in km
    dist = great_circle((center_lat, center_lon), (lat, lon)).km
    # bearing relative to north
    dlon = np.radians(lon - center_lon)
    lat1, lat2 = np.radians(center_lat), np.radians(lat)
    y = np.sin(dlon) * np.cos(lat2)
    x = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    bearing = (np.degrees(np.arctan2(y, x)) + 360) % 360
    # Convert to x/y in km (north = y, east = x)
    x_km = dist * np.sin(np.radians(bearing))
    y_km = dist * np.cos(np.radians(bearing))
    return x_km, y_km

df["x_km"], df["y_km"] = zip(*df.apply(lambda r: calc_offset(r["Lat"], r["Lon"], center_lat, center_lon), axis=1))

# --- Rotation for shear-relative coordinates ---
# --- Rotation for shear-relative coordinates ---
shear_dir = 250  # shear coming *from* 250° (blowing toward 70°)

# We want to rotate coordinates so that shear points UP.
# So, we rotate by (270 - shear_dir), which aligns shear direction with +y.
rotation_angle = 270 - shear_dir  
rot_rad = np.radians(rotation_angle)

df["x_shear"] = df["x_km"] * np.cos(rot_rad) - df["y_km"] * np.sin(rot_rad)
df["y_shear"] = df["x_km"] * np.sin(rot_rad) + df["y_km"] * np.cos(rot_rad)

# --- Function to plot ---
# --- Define the colors for each plot ---
north_circle_color = 'lightblue'      # shaded circle for the "relative to north" plot
shear_circle_color = 'lightcoral'     # shaded circle for the "relative to shear" plot


# --- Updated plotting function ---
def plot_drops(df, xcol, ycol, title, ax, circle_color):
    ax.set_aspect("equal")
    ax.set_xlim(-300, 300)
    ax.set_ylim(-300, 300)
    
    # shaded 200 km circle
    circle = plt.Circle((0, 0), 200, color=circle_color, alpha=0.25)
    ax.add_patch(circle)
    
    # concentric range rings
    for r in range(100, 301, 100):
        ring = plt.Circle((0, 0), r, color='k', fill=False, linestyle='--', linewidth=0.7)
        ax.add_patch(ring)
    
    # plot dropsonde points
    ax.scatter(df[xcol], df[ycol], c='dodgerblue', edgecolor='k', s=60)
    ax.scatter(0, 0, c='red', s=100, marker='*')
    ax.set_xlabel("Distance [km]")
    ax.set_ylabel("Distance [km]")
    ax.set_title(title, fontsize=11)


# --- Create the figure ---
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

plot_drops(df, "x_km", "y_km", "20140915N1 – Relative to North", axes[0], north_circle_color)
plot_drops(df, "x_shear", "y_shear", f"20140915N1 – Relative to Shear ({shear_dir}°)", axes[1], shear_circle_color)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from geopy.distance import great_circle
import re

# --- File path ---
cat_file = "DATA/20140915N1.cat"

# --- Read .cat file into DataFrame ---
rows = []
with open(cat_file, "r") as f:
    for line in f:
        if re.match(r"\s*\d+\s+\d+", line):  # line starts with sequence number and serial
            parts = line.split()
            seq = int(parts[0])
            serial = parts[1]
            date_time = parts[2] + " " + parts[3]
            lat = float(parts[4])
            lon = float(parts[5])
            pressure = float(parts[6])
            comment = " ".join(parts[7:])
            rows.append([seq, serial, date_time, lat, lon, pressure, comment])

df = pd.DataFrame(rows, columns=["Seq", "Serial", "Date/Time", "Lat", "Lon", "Pressure", "Comment"])

# --- Identify center drop (based on comment keyword) ---
center_lat = 27.5
center_lon = 55.5


# --- Compute distances and bearings relative to the center ---
def calc_offset(lat, lon, center_lat, center_lon):
    # distance in km
    dist = great_circle((center_lat, center_lon), (lat, lon)).km
    # bearing relative to north
    dlon = np.radians(lon - center_lon)
    lat1, lat2 = np.radians(center_lat), np.radians(lat)
    y = np.sin(dlon) * np.cos(lat2)
    x = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    bearing = (np.degrees(np.arctan2(y, x)) + 360) % 360
    # Convert to x/y in km (north = y, east = x)
    x_km = dist * np.sin(np.radians(bearing))
    y_km = dist * np.cos(np.radians(bearing))
    return x_km, y_km

df["x_km"], df["y_km"] = zip(*df.apply(lambda r: calc_offset(r["Lat"], r["Lon"], center_lat, center_lon), axis=1))

# --- Rotation for shear-relative coordinates ---
shear_dir = 250  # example: shear from 250° (i.e., to 70°)
rotation_angle = -(shear_dir - 180)  # rotate so shear direction points up
rot_rad = np.radians(rotation_angle)

df["x_shear"] = df["x_km"] * np.cos(rot_rad) - df["y_km"] * np.sin(rot_rad)
df["y_shear"] = df["x_km"] * np.sin(rot_rad) + df["y_km"] * np.cos(rot_rad)

# --- Function to plot ---
north_circle_color = 'plum'      # shaded circle for the "relative to north" plot
shear_circle_color = 'navy'     # shaded circle for the "relative to shear" plot

def plot_drops(df, xcol, ycol, title, ax, circle_color):
    ax.set_aspect("equal")
    ax.set_xlim(-300, 300)
    ax.set_ylim(-300, 300)
    
    # shaded 200 km circle
    circle = plt.Circle((0, 0), 200, color=circle_color, alpha=0.25)
    ax.add_patch(circle)
    
    # concentric range rings
    for r in range(100, 301, 100):
        ring = plt.Circle((0, 0), r, color='k', fill=False, linestyle='--', linewidth=0.7)
        ax.add_patch(ring)
    
    # plot dropsonde points
    ax.scatter(df[xcol], df[ycol], c='dodgerblue', edgecolor='k', s=60)
    ax.scatter(0, 0, c='red', s=100, marker='*')
    ax.set_xlabel("Distance [km]")
    ax.set_ylabel("Distance [km]")
    ax.set_title(title, fontsize=11)


# --- Create the figure ---
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

plot_drops(df, "x_km", "y_km", "20140915N1 – Relative to North", axes[0], north_circle_color)
plot_drops(df, "x_shear", "y_shear", f"20140915N1 – Relative to Shear ({shear_dir}°)", axes[1], shear_circle_color)

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from geopy.distance import great_circle
import re

# --- File path ---
cat_file = "DATA/20140916H1.cat"

# --- Read .cat file into DataFrame ---
rows = []
with open(cat_file, "r") as f:
    for line in f:
        if re.match(r"\s*\d+\s+\d+", line):  # line starts with sequence number and serial
            parts = line.split()
            seq = int(parts[0])
            serial = parts[1]
            date_time = parts[2] + " " + parts[3]
            lat = float(parts[4])
            lon = float(parts[5])
            pressure = float(parts[6])
            comment = " ".join(parts[7:])
            rows.append([seq, serial, date_time, lat, lon, pressure, comment])

df = pd.DataFrame(rows, columns=["Seq", "Serial", "Date/Time", "Lat", "Lon", "Pressure", "Comment"])

# --- Identify center drop (based on comment keyword) ---
center_lat = 31.15
center_lon = 57.75


# --- Compute distances and bearings relative to the center ---
def calc_offset(lat, lon, center_lat, center_lon):
    # distance in km
    dist = great_circle((center_lat, center_lon), (lat, lon)).km
    # bearing relative to north
    dlon = np.radians(lon - center_lon)
    lat1, lat2 = np.radians(center_lat), np.radians(lat)
    y = np.sin(dlon) * np.cos(lat2)
    x = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    bearing = (np.degrees(np.arctan2(y, x)) + 360) % 360
    # Convert to x/y in km (north = y, east = x)
    x_km = dist * np.sin(np.radians(bearing))
    y_km = dist * np.cos(np.radians(bearing))
    return x_km, y_km

df["x_km"], df["y_km"] = zip(*df.apply(lambda r: calc_offset(r["Lat"], r["Lon"], center_lat, center_lon), axis=1))

# --- Rotation for shear-relative coordinates ---
shear_dir = 250  # example: shear from 250° (i.e., to 70°)
rotation_angle = -(shear_dir - 180)  # rotate so shear direction points up
rot_rad = np.radians(rotation_angle)

df["x_shear"] = df["x_km"] * np.cos(rot_rad) - df["y_km"] * np.sin(rot_rad)
df["y_shear"] = df["x_km"] * np.sin(rot_rad) + df["y_km"] * np.cos(rot_rad)

# --- Function to plot ---
north_circle_color = 'plum'      # shaded circle for the "relative to north" plot
shear_circle_color = 'navy'     # shaded circle for the "relative to shear" plot

def plot_drops(df, xcol, ycol, title, ax, circle_color):
    ax.set_aspect("equal")
    ax.set_xlim(-300, 300)
    ax.set_ylim(-300, 300)
    
    # shaded 200 km circle
    circle = plt.Circle((0, 0), 200, color=circle_color, alpha=0.25)
    ax.add_patch(circle)
    
    # concentric range rings
    for r in range(100, 301, 100):
        ring = plt.Circle((0, 0), r, color='k', fill=False, linestyle='--', linewidth=0.7)
        ax.add_patch(ring)
    
    # plot dropsonde points
    ax.scatter(df[xcol], df[ycol], c='dodgerblue', edgecolor='k', s=60)
    ax.scatter(0, 0, c='red', s=100, marker='*')
    ax.set_xlabel("Distance [km]")
    ax.set_ylabel("Distance [km]")
    ax.set_title(title, fontsize=11)


# --- Create the figure ---
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

plot_drops(df, "x_km", "y_km", "20140915N1 – Relative to North", axes[0], north_circle_color)
plot_drops(df, "x_shear", "y_shear", f"20140915N1 – Relative to Shear ({shear_dir}°)", axes[1], shear_circle_color)

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from geopy.distance import great_circle
import re

#The 14th around 14-17 UTC 

# --- File path ---
cat_file = "DATA/20140914I1.cat"

# --- Read .cat file into DataFrame ---
rows = []
with open(cat_file, "r") as f:
    for line in f:
        if re.match(r"\s*\d+\s+\d+", line):  # line starts with sequence number and serial
            parts = line.split()
            seq = int(parts[0])
            serial = parts[1]
            date_time = parts[2] + " " + parts[3]
            lat = float(parts[4])
            lon = float(parts[5])
            pressure = float(parts[6])
            comment = " ".join(parts[7:])
            rows.append([seq, serial, date_time, lat, lon, pressure, comment])

df = pd.DataFrame(rows, columns=["Seq", "Serial", "Date/Time", "Lat", "Lon", "Pressure", "Comment"])

# --- Identify center drop (based on comment keyword) ---
center_lat = 24.8
center_lon = 50.9


# --- Compute distances and bearings relative to the center ---
def calc_offset(lat, lon, center_lat, center_lon):
    # distance in km
    dist = great_circle((center_lat, center_lon), (lat, lon)).km
    # bearing relative to north
    dlon = np.radians(lon - center_lon)
    lat1, lat2 = np.radians(center_lat), np.radians(lat)
    y = np.sin(dlon) * np.cos(lat2)
    x = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    bearing = (np.degrees(np.arctan2(y, x)) + 360) % 360
    # Convert to x/y in km (north = y, east = x)
    x_km = dist * np.sin(np.radians(bearing))
    y_km = dist * np.cos(np.radians(bearing))
    return x_km, y_km

df["x_km"], df["y_km"] = zip(*df.apply(lambda r: calc_offset(r["Lat"], r["Lon"], center_lat, center_lon), axis=1))

# --- Rotation for shear-relative coordinates ---
shear_dir = 210  
rotation_angle = -(shear_dir - 180)  # rotate so shear direction points up
rot_rad = np.radians(rotation_angle)

df["x_shear"] = df["x_km"] * np.cos(rot_rad) - df["y_km"] * np.sin(rot_rad)
df["y_shear"] = df["x_km"] * np.sin(rot_rad) + df["y_km"] * np.cos(rot_rad)

# --- Function to plot ---
import random

color_choices = [
    'plum', 'navy', 'teal', 'darkorange', 'lightcoral', 'mediumseagreen',
    'gold', 'royalblue', 'tomato', 'orchid', 'slateblue', 'skyblue'
]

north_circle_color = random.choice(color_choices)
shear_circle_color = random.choice(color_choices)   # shaded circle for the "relative to shear" plot

def plot_drops(df, xcol, ycol, title, ax, circle_color, rotation_angle=0):
    ax.set_aspect("equal")
    ax.set_xlim(-300, 300)
    ax.set_ylim(-300, 300)
    
    # shaded 200 km circle
    circle = plt.Circle((0, 0), 200, color=circle_color, alpha=0.25)
    ax.add_patch(circle)
    
    # concentric rings
    for r in range(100, 301, 100):
        ring = plt.Circle((0, 0), r, color='k', fill=False, linestyle='--', linewidth=0.7)
        ax.add_patch(ring)
    
    # Dropsondes
    ax.scatter(df[xcol], df[ycol], c='dodgerblue', edgecolor='k', s=60, zorder=3)
    ax.scatter(0, 0, c='red', s=100, marker='*', zorder=4)
    ax.set_xlabel("Distance [km]")
    ax.set_ylabel("Distance [km]")
    ax.set_title(title, fontsize=11)

    # --- Add compass / north arrow ---
    compass_x, compass_y = -270, 270  # upper-left corner
    arrow_length = 40

    # Rotate the compass by the same amount as the coordinate rotation
    # (so it still points toward true north on the shear-relative plot)
    rot_rad = np.radians(rotation_angle)
    dx = arrow_length * np.sin(rot_rad)
    dy = arrow_length * np.cos(rot_rad)

    # Draw rotated arrow
    ax.arrow(
        compass_x, compass_y,
        dx, dy,
        head_width=15, head_length=15,
        fc='black', ec='black', lw=1.2, zorder=5
    )

    # Label "N" (positioned slightly beyond arrow tip)
    ax.text(
        compass_x + dx * 1.2,
        compass_y + dy * 1.2,
        "N", ha='center', va='center',
        fontsize=10, fontweight='bold'
    )


# --- Create the figure ---
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# Relative to North
plot_drops(df, "x_km", "y_km", "20140916H1 – Relative to North", axes[0], north_circle_color)

# Relative to Shear
plot_drops(df, "x_shear", "y_shear", f"20140916H1 – Relative to Shear ({shear_dir}°)", axes[1], shear_circle_color, rotation_angle)

plt.tight_layout()
plt.show()


In [ ]:
####BEST ONE YET

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from geopy.distance import great_circle
import re
import random

# --- File path ---
cat_file = "DATA/20140914I1.cat"

# --- Read .cat file into DataFrame ---
rows = []
with open(cat_file, "r") as f:
    for line in f:
        if re.match(r"\s*\d+\s+\d+", line):
            parts = line.split()
            seq = int(parts[0])
            serial = parts[1]
            date_time = parts[2] + " " + parts[3]
            lat = float(parts[4])
            lon = float(parts[5])
            pressure = float(parts[6])
            comment = " ".join(parts[7:])
            rows.append([seq, serial, date_time, lat, lon, pressure, comment])

df = pd.DataFrame(rows, columns=["Seq", "Serial", "Date/Time", "Lat", "Lon", "Pressure", "Comment"])

# --- Identify center drop ---
center_lat = 24.8
center_lon = 50.9

# --- Compute offsets relative to the center ---
def calc_offset(lat, lon, center_lat, center_lon):
    dist = great_circle((center_lat, center_lon), (lat, lon)).km
    dlon = np.radians(lon - center_lon)
    lat1, lat2 = np.radians(center_lat), np.radians(lat)
    y = np.sin(dlon) * np.cos(lat2)
    x = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    bearing = (np.degrees(np.arctan2(y, x)) + 360) % 360
    x_km = dist * np.sin(np.radians(bearing))
    y_km = dist * np.cos(np.radians(bearing))
    return x_km, y_km

df["x_km"], df["y_km"] = zip(*df.apply(lambda r: calc_offset(r["Lat"], r["Lon"], center_lat, center_lon), axis=1))

# --- Rotation for shear-relative coordinates ---
# --- Rotation for shear-relative coordinates ---
shear_dir = 210  # shear coming *from* 210° (i.e., toward 30°)

# Rotate coordinates so shear points UP (aligned with +y axis)
rotation_angle = 270 - shear_dir  
rot_rad = np.radians(rotation_angle)

df["x_shear"] = df["x_km"] * np.cos(rot_rad) - df["y_km"] * np.sin(rot_rad)
df["y_shear"] = df["x_km"] * np.sin(rot_rad) + df["y_km"] * np.cos(rot_rad)

# --- Random colors ---
color_choices = [
    'plum', 'navy', 'teal', 'darkorange', 'lightcoral', 'mediumseagreen',
    'gold', 'royalblue', 'tomato', 'orchid', 'slateblue', 'skyblue'
]
north_circle_color = random.choice(color_choices)
shear_circle_color = random.choice(color_choices)

# --- Plotting function ---
def plot_drops(df, xcol, ycol, title, ax, circle_color, rotation_angle=0):
    ax.set_aspect("equal")
    ax.set_xlim(-300, 300)
    ax.set_ylim(-300, 300)
    
    # Shaded 200 km circle
    ax.add_patch(plt.Circle((0, 0), 200, color=circle_color, alpha=0.25))
    
    # Concentric rings
    for r in range(100, 301, 100):
        ax.add_patch(plt.Circle((0, 0), r, color='k', fill=False, linestyle='--', linewidth=0.7))
    
    # Dropsondes
    ax.scatter(df[xcol], df[ycol], c='dodgerblue', edgecolor='k', s=60, zorder=3)
    ax.scatter(0, 0, c='red', s=100, marker='*', zorder=4)
    ax.set_xlabel("Distance [km]")
    ax.set_ylabel("Distance [km]")
    ax.set_title(title, fontsize=11)
    
    # --- Add compass / north arrow ---
    compass_x, compass_y = -270, 270
    arrow_length = 40
    
    # For shear plot, rotate arrow opposite to the shear rotation
    rot_rad = np.radians(-rotation_angle)
    dx = arrow_length * np.sin(rot_rad)
    dy = arrow_length * np.cos(rot_rad)
    
    ax.arrow(compass_x, compass_y, dx, dy, head_width=15, head_length=15,
             fc='black', ec='black', lw=1.2, zorder=5)
    ax.text(compass_x + dx*1.2, compass_y + dy*1.2, "N",
            ha='center', va='center', fontsize=10, fontweight='bold')

# --- Create figure ---
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# Relative to North
plot_drops(df, "x_km", "y_km", "20140916H1 – Relative to North", axes[0], north_circle_color)

# Relative to Shear
plot_drops(df, "x_shear", "y_shear", f"20140916H1 – Relative to Shear ({shear_dir}°)", axes[1], shear_circle_color, rotation_angle)

plt.tight_layout()
plt.show()
